# MNIST Inverse Problem — HMC vs Annealed HMC vs SMC

**Prerequisites**: Run `python train.py` first to produce `checkpoints/vae.pt` and `checkpoints/flow.pt`.

## Problem setup
- **Prior** `p(z)`: MNIST digit distribution in a 32-dim VAE latent space, modelled by a RealNVP flow.
- **Decoder** `D`: VAE decoder, `z (32,) → x (784,)`.
- **Forward model**: `y = A x + noise` where `A` keeps only the **right half** of each image (392 pixels observed, 392 unknown).
- **Goal**: sample `p(z|y)` — all 32-dim latent codes that decode to digits consistent with the observed half.

## MCMC happens in ε-space
```
ε (32,) → flow.inverse → z (32,) → vae.decode → x (784,) → A → y (392,)
log p(ε|y) = −½‖ε‖² − ½‖y − A(D(G(ε)))‖² / σ²
```
Same formula as the 2D case — only the chain is longer.

## Three samplers
| Method | Mode mixing? | Exact? |
|--------|-------------|--------|
| **(a) Vanilla HMC** | ✗ single chain, gets trapped | ✓ if mixes |
| **(b) Annealed HMC** | ✓ better — broad likelihood smooths barriers | ✗ biased |
| **(c) SMC** | ✓ particle ensemble covers all modes from β=0 | ✓ asymptotically |

In [ ]:
import sys, os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ── local modules ──────────────────────────────────────────────────────────────
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('../new_toy_2d'))   # reuse LogPosterior + samplers

from config import *
from vae import VAE
from flow_model import RealNVP
from forward_models import make_inpainting_op, make_observation, overlay_observation
from log_posterior import LogPosterior
from samplers import (
    HMCSampler, SMCSampler,
    run_annealed_hmc,
    compute_ess,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
torch.manual_seed(0)
np.random.seed(0)

---
## Step 1 — Load trained models

In [ ]:
# ── VAE ────────────────────────────────────────────────────────────────────────
vae_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'vae.pt'), map_location=device)
vae = VAE(latent_dim=vae_ckpt['latent_dim'], hidden_dim=vae_ckpt['hidden_dim']).to(device)
vae.load_state_dict(vae_ckpt['model_state'])
vae.eval()
print(f'VAE loaded  (latent_dim={vae_ckpt["latent_dim"]},  final_loss={vae_ckpt["final_loss"]:.4f})')

# ── Flow ───────────────────────────────────────────────────────────────────────
flow_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'flow.pt'), map_location=device)
flow = RealNVP(dim=flow_ckpt['dim'], hidden_dim=flow_ckpt['hidden_dim'],
               n_layers=flow_ckpt['n_layers']).to(device)
flow.load_state_dict(flow_ckpt['model_state'])
flow.eval()
print(f'Flow loaded (dim={flow_ckpt["dim"]}, layers={flow_ckpt["n_layers"]}, final_NLL={flow_ckpt["final_nll"]:.4f})')

# ── Callables for LogPosterior ─────────────────────────────────────────────────
# flow_inverse: ε (32,) → z (32,)
flow_inverse = lambda eps: flow.inverse(eps.unsqueeze(0)).squeeze(0)
# decoder:      z (32,) → x (784,)
decoder = lambda z: vae.decode(z.unsqueeze(0)).squeeze(0)

In [ ]:
# Quick sanity: generate 16 random digits
with torch.no_grad():
    eps_rand = torch.randn(16, LATENT_DIM, device=device)
    z_rand   = flow.inverse(eps_rand)
    x_rand   = vae.decode(z_rand).cpu().numpy()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_rand[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
plt.suptitle('Prior samples: ε ~ N(0,I) → flow.inverse → vae.decode', fontsize=11)
plt.tight_layout()
plt.show()

---
## Step 2 — Set Up Inverse Problem (Inpainting)

In [ ]:
# ── Forward operator ───────────────────────────────────────────────────────────
forward_op, obs_dim, mask_flat = make_inpainting_op(MASK_TYPE, device=device)
print(f'Mask type   : {MASK_TYPE}')
print(f'obs_dim     : {obs_dim}  (out of 784)')

# ── Pick a test image ──────────────────────────────────────────────────────────
# Load MNIST test set and find an ambiguous digit.
# We look for an image where the unobserved half is non-trivial — ideally one
# where multiple completions are plausible (e.g. the right half of a '3' looks
# similar to an '8' or '5').
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),
])
test_set = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)

# Pick digit class 3 (often ambiguous with 8 under right-half mask)
TARGET_CLASS = 3
SAMPLE_IDX   = 5    # change this to try different examples

candidates = [(x, y) for x, y in test_set if y == TARGET_CLASS]
x_true_np, label = candidates[SAMPLE_IDX]
x_true = x_true_np.to(device)   # (784,)

# Observed measurement
y_obs = make_observation(forward_op, x_true, SIGMA_N, seed=42)

print(f'\nTest digit  : {label}')
print(f'x_true range: [{x_true.min():.3f}, {x_true.max():.3f}]')
print(f'y_obs shape : {y_obs.shape}')

In [ ]:
# Visualise the observation
obs_img     = overlay_observation(x_true, mask_flat, fill=0.5)
x_true_img  = x_true.cpu().view(28, 28).numpy()

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(x_true_img,  cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Ground truth (digit {label})')
axes[0].axis('off')
axes[1].imshow(obs_img, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Observed y  (mask={MASK_TYPE}, grey=unknown)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Build log-posterior and shared ε initialisation ───────────────────────────
log_posterior = LogPosterior(
    flow_inverse = flow_inverse,
    decoder      = decoder,
    forward_op   = forward_op,
    y            = y_obs,
    sigma_n      = SIGMA_N,
)

# Initialise ε at the encoding of the observed reconstruction
# (map x_true → z → ε as a warm start; in practice any point works)
with torch.no_grad():
    z_init, _ = flow.forward(vae.encode(x_true.unsqueeze(0)))
    eps_init  = z_init.squeeze(0).detach()
print(f'eps_init norm: {eps_init.norm():.3f}')

---
## Step 3a — (a) Vanilla HMC

In [ ]:
hmc = HMCSampler(log_posterior, dim=LATENT_DIM, step_size=None,
                 n_leapfrog=N_HMC_LEAPFROG)
eps_hmc, info_hmc = hmc.sample(eps_init.clone(), n_samples=N_HMC_SAMPLES,
                                n_warmup=N_HMC_WARMUP, verbose=True)

with torch.no_grad():
    x_hmc = vae.decode(flow.inverse(eps_hmc)).cpu().numpy()   # (N, 784)

ess_hmc = compute_ess(eps_hmc)
print(f'\nAcceptance rate : {info_hmc["acceptance_rate"]:.3f}')
print(f'ESS (min over d): {ess_hmc.min():.0f}')

---
## Step 3b — (b) Annealed HMC
Geometrically-spaced σ schedule from `ANN_SIGMA_START` → `SIGMA_N`.
Each stage uses the last sample of the previous stage as its warm start.

In [ ]:
import numpy as np

sigma_schedule = np.geomspace(ANN_SIGMA_START, SIGMA_N, N_ANN_STAGES).tolist()
print(f'Sigma schedule ({N_ANN_STAGES} stages):')
print('  ' + ' → '.join(f'{s:.4f}' for s in sigma_schedule))

def log_posterior_factory(sigma_eff):
    return LogPosterior(
        flow_inverse = flow_inverse,
        decoder      = decoder,
        forward_op   = forward_op,
        y            = y_obs,
        sigma_n      = sigma_eff,
    )

In [ ]:
annealed = run_annealed_hmc(
    log_posterior_factory = log_posterior_factory,
    eps_init              = eps_init.clone(),
    sigma_schedule        = sigma_schedule,
    n_samples_per_stage   = N_ANN_SAMPLES_PER_STAGE,
    n_warmup_per_stage    = N_ANN_WARMUP_PER_STAGE,
    n_leapfrog            = N_ANN_LEAPFROG,
    verbose               = True,
)

eps_ann = annealed['final_samples']
with torch.no_grad():
    x_ann = vae.decode(flow.inverse(eps_ann)).cpu().numpy()

ess_ann = compute_ess(eps_ann)
print(f'\nFinal ESS (min): {ess_ann.min():.0f}')

---
## Step 3c — (c) SMC
Particles initialised at ε ~ N(0,I) → all modes of the prior are represented from the start.
Adaptive β schedule gradually turns on the likelihood; resampling prevents collapse.

In [ ]:
def log_likelihood_fn(eps):
    """Only the likelihood term (prior handled separately by SMC)."""
    z   = flow_inverse(eps)
    x   = decoder(z)
    Ax  = forward_op(x)
    return -0.5 * ((y_obs - Ax) ** 2).sum() / (SIGMA_N ** 2)

smc = SMCSampler(
    log_likelihood_fn = log_likelihood_fn,
    dim               = LATENT_DIM,
    n_particles       = N_SMC_PARTICLES,
    n_mcmc_steps      = N_SMC_MCMC_STEPS,
    n_leapfrog        = N_SMC_LEAPFROG,
    ess_threshold     = SMC_ESS_THRESHOLD,
)

smc_result = smc.sample(device=device, verbose=True)
eps_smc    = smc_result['samples']

with torch.no_grad():
    x_smc = vae.decode(flow.inverse(eps_smc)).cpu().numpy()

print(f'\nSMC: {smc_result["n_stages"]} stages')
print(f'log p(y) estimate: {smc_result["log_normalizer"]:.3f}')

---
## Step 4 — 1 × 3 Comparison

In [ ]:
def make_montage(images, nrow=4, ncol=4, img_size=28):
    """Stack (N, 784) samples into a (nrow*28, ncol*28) grid image."""
    canvas = np.zeros((nrow * img_size, ncol * img_size))
    for idx in range(min(nrow * ncol, len(images))):
        r, c = divmod(idx, ncol)
        canvas[r*img_size:(r+1)*img_size, c*img_size:(c+1)*img_size] = \
            images[idx].reshape(img_size, img_size)
    return canvas

NROW, NCOL = 4, 4   # 16 samples per panel

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.25,
                        height_ratios=[1, 4])

configs = [
    ('(a) HMC\n(mode collapse)',        x_hmc,  'Oranges'),
    ('(b) Annealed HMC\n(auto σ sched)', x_ann, 'Blues'),
    ('(c) SMC\n(exact, unbiased)',       x_smc,  'Greens'),
]

# ── Row 0: observed input (same for all panels) ────────────────────────────────
for col in range(3):
    ax = fig.add_subplot(gs[0, col])
    ax.imshow(obs_img, cmap='gray', vmin=0, vmax=1)
    ax.set_title('Observed y  (grey = unknown)', fontsize=9)
    ax.axis('off')

# ── Row 1: 4×4 grid of posterior samples ──────────────────────────────────────
for col, (title, x_s, cmap) in enumerate(configs):
    ax   = fig.add_subplot(gs[1, col])
    mont = make_montage(x_s[:NROW*NCOL], nrow=NROW, ncol=NCOL)
    ax.imshow(mont, cmap='gray', vmin=0, vmax=1)

    # Diversity metric: std of pixel values across samples
    pixel_std = x_s[:NROW*NCOL].std(axis=0).mean()
    ax.set_title(f'{title}\npixel std={pixel_std:.3f}', fontsize=11, fontweight='bold')
    ax.axis('off')

    # Draw grid lines between samples
    for i in range(1, NROW):
        ax.axhline(i * 28 - 0.5, color='white', lw=0.5)
    for j in range(1, NCOL):
        ax.axvline(j * 28 - 0.5, color='white', lw=0.5)

fig.suptitle(
    f'MNIST Inpainting Posterior  —  digit={label},  mask={MASK_TYPE},  σ_n={SIGMA_N}\n'
    f'Each panel: 16 posterior samples decoded to pixel space  (higher pixel_std = more diverse)',
    fontsize=12
)
plt.savefig('mnist_comparison_1x3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: mnist_comparison_1x3.png')

In [ ]:
# ── SMC diagnostics ────────────────────────────────────────────────────────────
diags    = smc_result['diagnostics']
stages   = [d['stage']      for d in diags]
betas    = [d['beta']       for d in diags]
ess_f    = [d['ess_frac']   for d in diags]
acc      = [d['acceptance'] for d in diags]
resample = [d['stage'] for d in diags if d['resampled']]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].step(stages, betas, where='post', lw=2, color='steelblue')
axes[0].set(xlabel='Stage', ylabel='β', title='Adaptive β schedule')
axes[0].grid(True, alpha=0.3)

axes[1].plot(stages, ess_f, 'orange', lw=1.5)
axes[1].axhline(smc.ess_threshold, color='red', ls=':', label='threshold')
for s in resample:
    axes[1].axvline(s, color='red', alpha=0.3, lw=0.8)
axes[1].set(xlabel='Stage', ylabel='ESS/N', title='ESS fraction (red = resample)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(stages, acc, 'green', lw=1.5)
axes[2].axhline(0.6, color='gray', ls=':', alpha=0.6, label='target 0.6')
axes[2].set(xlabel='Stage', ylabel='acceptance', title='HMC acceptance per stage')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Summary

| | HMC | Annealed HMC | SMC |
|---|---|---|---|
| **Finds multiple completions** | ✗ — all 16 samples look identical | Sometimes | ✓ — visually diverse samples |
| **Exact posterior** | ✓ (if mixed) | ✗ biased | ✓ asymptotically |
| **log p(y) estimate** | ✗ | ✗ | ✓ |
| **Diversity metric** | low pixel_std | medium | high pixel_std |

### Why SMC works
SMC initialises all particles from `ε ~ N(0,I)` — the full prior over MNIST digits.
At β=0 every digit is equally likely. As β increases, particles with reconstructions
inconsistent with `y` are down-weighted and eventually resampled away, but the ensemble
never loses coverage of any mode. HMC starts from a single point and gets trapped.

### Key takeaway for the thesis
The ε-space reformulation (NSPS) does **not** solve mode mixing — it only makes the
prior evaluation cheap. SMC solves mode mixing. The two ideas are orthogonal and
complementary: NSPS makes each likelihood evaluation fast, SMC makes the sampler
globally correct.